In [9]:
import pandas as pd

# -----------------------------
# Load data
# -----------------------------
prices = pd.read_csv(
    "../data/processed/AAPL_features.csv",
    parse_dates=["Date"]
)

sentiment = pd.read_csv(
    "../data/processed/daily/AAPL_daily_sentiment.csv",
    parse_dates=["Date"]
)

# Drop leftover index column if present
sentiment = sentiment.drop(columns=["Unnamed: 0"], errors="ignore")

# Normalize dates to timezone-naive, date-only so the merge keys match
prices["Date"] = pd.to_datetime(prices["Date"]).dt.tz_localize(None).dt.normalize()
sentiment["Date"] = (
    pd.to_datetime(sentiment["Date"], utc=True).dt.tz_localize(None).dt.normalize()
)

# -----------------------------
# Ensure same column names
# -----------------------------
prices.rename(columns={"Ticker": "Stock_symbol"}, inplace=True)

# Collapse any intraday duplicate sentiment rows into one row per day
sentiment = (
    sentiment.groupby(["Date", "Stock_symbol"], as_index=False)
    .agg({"AvgSentiment": "mean", "AvgConfidence": "mean", "HeadlineCount": "sum"})
)

# -----------------------------
# Merge
# Keep ALL trading days
# -----------------------------
training = pd.merge(
    prices,
    sentiment,
    on=["Date", "Stock_symbol"],
    how="left"
)

# -----------------------------
# Fill missing values
# -----------------------------
training["HeadlineCount"] = training["HeadlineCount"].fillna(0)

sentiment_columns = [
    "AvgSentiment",
    "AvgConfidence",
]

training[sentiment_columns] = training[sentiment_columns].fillna(0)

# -----------------------------
# Create target variable
# -----------------------------
training["Tomorrow_Close"] = training["Close"].shift(-1)

training["TomorrowUp"] = (
    training["Tomorrow_Close"] > training["Close"]
).astype(int)

# Remove last row (no tomorrow price)
training = training.iloc[:-1]

# -----------------------------
# Verify missing values
# -----------------------------
print(training.isnull().sum())

# -----------------------------
# Save
# -----------------------------
training.to_csv(
    "../data/processed/AAPL_training_dataset.csv",
    index=False
)

training.head()

Date                        0
Stock_symbol                0
Open                        0
High                        0
Low                         0
Close                       0
Volume                      0
Daily_Return                0
SMA_10                      0
SMA_20                      0
EMA_10                      0
EMA_20                      0
RSI_14                      0
MACD                        0
MACD_signal                 0
MACD_hist                   0
Volatility_20               0
Volatility_20_annualized    0
AvgSentiment                0
HeadlineCount               0
AvgConfidence               0
Tomorrow_Close              0
TomorrowUp                  0
dtype: int64


,Date,Stock_symbol,Open,High,Low,Close,Volume,Daily_Return,SMA_10,SMA_20,...,MACD,MACD_signal,MACD_hist,Volatility_20,Volatility_20_annualized,AvgSentiment,HeadlineCount,AvgConfidence,Tomorrow_Close,TomorrowUp
0,2020-04-06,AAPL,60.568150,63.515685,60.201219,63.361191,201820400,0.087238,60.288112,60.646595,...,-0.796200,-1.090492,0.294291,0.065030,1.032323,0.400000,5.0,0.919917,62.627316,0
1,2020-04-07,AAPL,65.372072,65.589342,62.523514,62.627316,202887200,-0.011582,60.591073,60.333857,...,-0.552620,-0.982917,0.430297,0.062898,0.998474,0.500000,2.0,0.924318,64.230209,1
2,2020-04-08,AAPL,63.426332,64.544031,63.061818,64.230209,168895200,0.025594,61.087155,60.220878,...,-0.227617,-0.831857,0.604240,0.062736,0.995899,0.100000,10.0,0.803138,64.693726,1
3,2020-04-09,AAPL,64.865127,65.195849,63.899513,64.693726,161834800,0.007216,61.317695,60.459385,...,0.066585,-0.652169,0.718754,0.058258,0.924819,0.200000,5.0,0.764581,65.963516,1
4,2020-04-13,AAPL,64.770982,66.072151,64.172299,65.963516,131022800,0.019628,61.933516,60.402415,...,0.397620,-0.442211,0.839831,0.051864,0.823312,-0.166667,6.0,0.804346,69.294876,1


In [10]:
import os

# -----------------------------
# Merge prices + sentiment for ALL stocks
# -----------------------------
TICKERS = ["AAPL", "AMZN", "GOOGL", "META", "MSFT", "NVDA", "TSLA"]


def build_training_dataset(ticker: str) -> pd.DataFrame | None:
    features_path = f"../data/processed/{ticker}_features.csv"
    sentiment_path = f"../data/processed/daily/{ticker}_daily_sentiment.csv"

    if not os.path.exists(features_path):
        print(f"[skip] {ticker}: no features file")
        return None

    prices = pd.read_csv(features_path, parse_dates=["Date"])
    prices.rename(columns={"Ticker": "Stock_symbol"}, inplace=True)
    prices["Date"] = pd.to_datetime(prices["Date"]).dt.tz_localize(None).dt.normalize()

    if os.path.exists(sentiment_path):
        sentiment = pd.read_csv(sentiment_path, parse_dates=["Date"])
        sentiment = sentiment.drop(columns=["Unnamed: 0"], errors="ignore")
        sentiment["Date"] = (
            pd.to_datetime(sentiment["Date"], utc=True)
            .dt.tz_localize(None)
            .dt.normalize()
        )
        # collapse any intraday duplicate rows into one row per day
        sentiment = (
            sentiment.groupby(["Date", "Stock_symbol"], as_index=False)
            .agg({"AvgSentiment": "mean", "AvgConfidence": "mean", "HeadlineCount": "sum"})
        )
        training = pd.merge(prices, sentiment, on=["Date", "Stock_symbol"], how="left")
    else:
        print(f"[warn] {ticker}: no sentiment file, filling sentiment columns with 0")
        training = prices.copy()
        training["AvgSentiment"] = 0
        training["AvgConfidence"] = 0
        training["HeadlineCount"] = 0

    # fill missing sentiment values
    training["HeadlineCount"] = training["HeadlineCount"].fillna(0)
    training[["AvgSentiment", "AvgConfidence"]] = training[
        ["AvgSentiment", "AvgConfidence"]
    ].fillna(0)

    # create target variable (per-stock so the shift is correct)
    training = training.sort_values("Date").reset_index(drop=True)
    training["Tomorrow_Close"] = training["Close"].shift(-1)
    training["TomorrowUp"] = (
        training["Tomorrow_Close"] > training["Close"]
    ).astype(int)
    training = training.iloc[:-1]

    out_path = f"../data/processed/{ticker}_training_dataset.csv"
    training.to_csv(out_path, index=False)
    print(f"[ok] {ticker}: {len(training)} rows -> {out_path}")
    return training

all_training = {}
for t in TICKERS:
    df = build_training_dataset(t)
    if df is not None:
        all_training[t] = df

# Combined dataset across all stocks
combined = pd.concat(all_training.values(), ignore_index=True)
combined.to_csv("../data/processed/all_training_dataset.csv", index=False)
print("Combined dataset shape:", combined.shape)
combined.head() 

[ok] AAPL: 45 rows -> ../data/processed/AAPL_training_dataset.csv
[ok] AMZN: 11 rows -> ../data/processed/AMZN_training_dataset.csv
[ok] GOOGL: 452 rows -> ../data/processed/GOOGL_training_dataset.csv
[warn] META: no sentiment file, filling sentiment columns with 0
[ok] META: 1232 rows -> ../data/processed/META_training_dataset.csv
[warn] MSFT: no sentiment file, filling sentiment columns with 0
[ok] MSFT: 1232 rows -> ../data/processed/MSFT_training_dataset.csv
[ok] NVDA: 2313 rows -> ../data/processed/NVDA_training_dataset.csv
[ok] TSLA: 218 rows -> ../data/processed/TSLA_training_dataset.csv
Combined dataset shape: (5503, 23)


,Date,Stock_symbol,Open,High,Low,Close,Volume,Daily_Return,SMA_10,SMA_20,...,MACD,MACD_signal,MACD_hist,Volatility_20,Volatility_20_annualized,AvgSentiment,AvgConfidence,HeadlineCount,Tomorrow_Close,TomorrowUp
0,2020-04-06,AAPL,60.568150,63.515685,60.201219,63.361191,201820400,0.087238,60.288112,60.646595,...,-0.796200,-1.090492,0.294291,0.065030,1.032323,0.400000,0.919917,5.0,62.627316,0
1,2020-04-07,AAPL,65.372072,65.589342,62.523514,62.627316,202887200,-0.011582,60.591073,60.333857,...,-0.552620,-0.982917,0.430297,0.062898,0.998474,0.500000,0.924318,2.0,64.230209,1
2,2020-04-08,AAPL,63.426332,64.544031,63.061818,64.230209,168895200,0.025594,61.087155,60.220878,...,-0.227617,-0.831857,0.604240,0.062736,0.995899,0.100000,0.803138,10.0,64.693726,1
3,2020-04-09,AAPL,64.865127,65.195849,63.899513,64.693726,161834800,0.007216,61.317695,60.459385,...,0.066585,-0.652169,0.718754,0.058258,0.924819,0.200000,0.764581,5.0,65.963516,1
4,2020-04-13,AAPL,64.770982,66.072151,64.172299,65.963516,131022800,0.019628,61.933516,60.402415,...,0.397620,-0.442211,0.839831,0.051864,0.823312,-0.166667,0.804346,6.0,69.294876,1
